### Build & Push Docker Image

Build the training Docker image for `linux/amd64` (Vast.ai runs x86_64) and push to Docker Hub. 

**Note:** This cross-compilation build may take 30-60 minutes on Apple Silicon. The CUDA base image is ~8GB and llama.cpp compiles from source.

In [ ]:
import subprocess, sys, os

DOCKER_USERNAME = os.environ.get("DOCKER_USERNAME")
if not DOCKER_USERNAME:
    raise RuntimeError("DOCKER_USERNAME not found in environment. Check your .env file.")

IMAGE_NAME = f"{DOCKER_USERNAME}/2048-train:latest"
REPO_ROOT = os.getcwd()

# Persist buildx cache on disk so subsequent builds reuse layers.
# First build: ~30-60 min. Rebuilds (only changed layers): ~1-5 min.
CACHE_DIR = os.path.join(REPO_ROOT, ".docker-build-cache")
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"Building {IMAGE_NAME} for linux/amd64...")
print("First build: ~30-60 min. Rebuilds with cache: ~1-5 min.\n")

result = subprocess.run(
    [
        "docker", "buildx", "build",
        "--platform", "linux/amd64",
        "--cache-to", f"type=local,mode=max,dest={CACHE_DIR}",
        "--cache-from", f"type=local,src={CACHE_DIR}",
        "-t", IMAGE_NAME,
        "--load",
        ".",
    ],
    cwd=REPO_ROOT,
)

if result.returncode == 0:
    print(f"\nBuild succeeded: {IMAGE_NAME}")
else:
    raise RuntimeError(f"Build failed with exit code {result.returncode}")

In [ ]:
print(f"Pushing {IMAGE_NAME} to Docker Hub...")
result = subprocess.run(
    ["docker", "push", IMAGE_NAME],
)

if result.returncode == 0:
    print(f"\nPush succeeded: {IMAGE_NAME}")
else:
    raise RuntimeError(f"Push failed with exit code {result.returncode}")

### Launch Instance

Search for RTX 4090 interruptable instances and launch with our Docker image. The container automatically runs training + GGUF export via its CMD.

In [ ]:
from vastai import VastAI

# Uses VAST_API_KEY environment variable
vast = VastAI()

print("Searching for an RTX 4090 instance...\n")

# Search for RTX 4090 interruptable instances, sorted cheapest first
# Note: RTX 5090 (Blackwell/sm_120) is NOT supported by ART's vLLM fork
offers = vast.search_offers(
    query="gpu_name=RTX_4090 num_gpus=1 verified=true direct_port_count>=1 rentable=true",
    order="dph_total+",
    limit="20",
)

if not offers:
    raise RuntimeError("No RTX 4090 offers found. Try broadening the search criteria.")

# Filter out Chinese hosts — they often have poor connectivity to HuggingFace/W&B
def is_excluded(offer):
    geo = (offer.get("geolocation") or "").lower()
    country = (offer.get("country") or "").lower()
    for kw in ("china", "cn", "beijing", "shanghai", "shenzhen", "guangzhou"):
        if kw in geo or kw in country:
            return True
    return False

non_cn_offers = [o for o in offers if not is_excluded(o)]
if not non_cn_offers:
    print("Warning: all offers are from Chinese hosts. Proceeding anyway.")
    non_cn_offers = offers

# Pick the cheapest non-excluded offer
offer = non_cn_offers[0]
OFFER_ID = offer["id"]
print(f"Selected offer: {OFFER_ID}")
print(f"  GPU: {offer.get('gpu_name', 'N/A')}")
print(f"  Price: ${offer.get('dph_total', 'N/A')}/hr")
print(f"  Host: {offer.get('machine_id', 'N/A')}")
print(f"  Location: {offer.get('geolocation', 'N/A')}")

# Load secrets (dotenv already loaded in first cell)
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("HF_TOKEN not found in environment. Required for gated Qwen 3 model. Check your .env file.")

wandb_key = os.environ.get("WANDB_API_KEY")
if wandb_key:
    print("WANDB_API_KEY found — training metrics will be logged.")
else:
    print("WARNING: WANDB_API_KEY not set. Training will work but metrics won't be logged to W&B.")

# Build env dict — always include HF_TOKEN, optionally WANDB
env_vars = {
    "OUTPUT_DIR": "/app/output",
    "HF_HOME": "/app/output/.hf_cache",
    "HF_TOKEN": hf_token,
}
if wandb_key:
    env_vars["WANDB_API_KEY"] = wandb_key

instance = vast.create_instance(
    OFFER_ID,
    image=IMAGE_NAME,
    disk=80,
    env=env_vars,
    label="2048-train",
    runtype="args",  # Use Docker CMD directly — no SSH/Jupyter overhead
)

# Extract instance ID from API response
if isinstance(instance, dict):
    INSTANCE_ID = (
        instance.get("new_contract")
        or instance.get("new_instance", {}).get("id")
        or instance.get("id")
    )
    if not INSTANCE_ID:
        raise RuntimeError(f"Could not extract instance ID from response: {instance}")
elif isinstance(instance, (int, str)):
    INSTANCE_ID = instance
else:
    raise RuntimeError(f"Unexpected instance response: {instance}")

print(f"\nInstance created: {INSTANCE_ID}")
print("\nThe container will automatically run training + export on startup.")
print("\nObservability:")
print(f"  Logs:   vastai logs {INSTANCE_ID}")

### Monitor Training

Watch the training logs in real-time. Training runs 20 steps x 18 games (~40 min on RTX 4090), then export (~15 min).

In [ ]:
# Check instance status
status = vast.show_instance(INSTANCE_ID)
print(f"Instance status: {status.get('cur_state', 'N/A')}")
print(f"Image status: {status.get('image_state', 'N/A')}")

# Get recent logs
logs = vast.logs(INSTANCE_ID)
print(f"\n--- Recent logs ---\n{logs}")

### Download GGUF & Cleanup

Once training and export are complete, download the quantized GGUF model (~5.5GB) and destroy the instance to stop billing.

In [ ]:
import yaml, json

HF_REPO = os.environ.get("HF_REPO", "nathanpua/qwen3-8b-2048-gguf")
LMSTUDIO_DIR = os.path.expanduser("~/.lmstudio/models/lmstudio-community/Qwen3-8B-2048-GGUF")
os.makedirs(LMSTUDIO_DIR, exist_ok=True)
LOCAL_GGUF = os.path.join(LMSTUDIO_DIR, "qwen3-8b-2048-Q4_K_M.gguf")

print(f"Downloading GGUF from HuggingFace Hub ({HF_REPO})...")
hf_hub_download(
    repo_id=HF_REPO,
    filename="qwen3-8b-2048-Q4_K_M.gguf",
    local_dir=LMSTUDIO_DIR,
)
size_gb = os.path.getsize(LOCAL_GGUF) / (1024**3)
print(f"Downloaded to: {LOCAL_GGUF}")
print(f"File size: {size_gb:.1f} GB")

# Create LM Studio model.yaml wrapper so the thinking toggle appears in the UI.
# Without this wrapper, side-loaded GGUFs don't get the "Enable Thinking" toggle
# even though the chat template already supports enable_thinking.
HUB_MODEL_DIR = os.path.expanduser("~/.lmstudio/hub/models/nathanpua/qwen3-8b-2048")
os.makedirs(HUB_MODEL_DIR, exist_ok=True)

model_yaml = {
    "model": "nathanpua/qwen3-8b-2048",
    "base": [
        {
            "key": "lmstudio-community/Qwen3-8B-2048-GGUF",
            "sources": [
                {"type": "huggingface", "user": "nathanpua", "repo": HF_REPO.split("/", 1)[1]},
            ],
        }
    ],
    "metadataOverrides": {
        "domain": "llm",
        "architectures": ["qwen3"],
        "compatibilityTypes": ["gguf"],
        "paramsStrings": ["8B"],
        "minMemoryUsageBytes": 4600000000,
        "contextLengths": [40960],
        "vision": False,
        "reasoning": True,
        "trainedForToolUse": True,
    },
    "config": {
        "operation": {
            "fields": [
                {"key": "llm.prediction.topKSampling", "value": 20},
                {"key": "llm.prediction.minPSampling", "value": {"checked": True, "value": 0}},
            ]
        }
    },
    "customFields": [
        {
            "key": "enableThinking",
            "displayName": "Enable Thinking",
            "description": "Controls whether the model will think before replying",
            "type": "boolean",
            "defaultValue": True,
            "effects": [{"type": "setJinjaVariable", "variable": "enable_thinking"}],
        }
    ],
}

manifest = {
    "type": "model",
    "owner": "nathanpua",
    "name": "qwen3-8b-2048",
    "dependencies": [
        {
            "type": "model",
            "purpose": "baseModel",
            "modelKeys": ["lmstudio-community/Qwen3-8B-2048-GGUF"],
            "sources": [
                {"type": "huggingface", "user": "nathanpua", "repo": HF_REPO.split("/", 1)[1]},
            ],
        }
    ],
    "revision": 1,
}

with open(os.path.join(HUB_MODEL_DIR, "model.yaml"), "w") as f:
    yaml.dump(model_yaml, f, default_flow_style=False, sort_keys=False)
with open(os.path.join(HUB_MODEL_DIR, "manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2)

print(f"\nLM Studio wrapper created at {HUB_MODEL_DIR}")
print("Restart LM Studio to see the model with the thinking toggle.")

In [ ]:
# Destroy the instance to stop billing
print(f"Destroying instance {INSTANCE_ID}...")
result = vast.destroy_instance(INSTANCE_ID)
print(f"Instance destroyed: {result}")

print("\nDone! Load the GGUF in LM Studio and run 2048-local.ipynb to evaluate.")

# WARNING: If this cell is not reached (e.g. kernel dies), the instance keeps billing.
# Run this manually if needed: vast.destroy_instance(INSTANCE_ID)